# Heisenberg spin-chain dynamics — and why the new scheduling matters

This notebook runs a 4-qubit 1D Heisenberg-model Trotter simulation through
the provider and uses it to compare the two timing policies:

- **`timing_policy="qiskit"`** (default) — the pulse schedule follows the
  transpiler's scheduled start times, so independent gates run in parallel
- **`timing_policy="legacy_device_gateway"`** (deprecated) — the
  device-gateway-compatible sequential construction, which serializes the
  circuit with barriers

Two improvements show up on a brickwork Trotter circuit:

1. **Parallelism** — the even-layer interactions on `Q01–Q00` and
   `Q03–Q02` run simultaneously under the new policy; the legacy path
   serializes them.
2. **Compatibility with scheduled circuits** — the modern flow
   (`scheduling_method="alap"`, DD passes, timeline analysis) produces
   circuits with explicit delays. The legacy path replays those delays
   on top of its serial schedule, silently inflating it by >2×.

We show the schedules side by side, quantify both effects, sweep the
evolution angle, and set up the hardware comparison (gated behind
`RUN_ON_HARDWARE`).

The model and observable follow the companion benchmark repository
[qiskit-qubex-bench](https://github.com/orangekame3/qiskit-qubex-bench)
(which adds DD variants, Aer noise models, and pulse-level simulation on
top of this protocol). The setup (offline `qubex.Experiment` from
[qubex-config/](qubex-config/)) is the same as in
[tutorial.ipynb](tutorial.ipynb).

Requirements:

```bash
pip install "qiskit-qubex-provider[qubex,plot] @ git+https://github.com/orangekame3/qiskit-qubex-provider.git"
```

In [1]:
import numpy as np
import plotly.graph_objects as go
from qiskit import QuantumCircuit, transpile
from qubex import Experiment

from qiskit_qubex_provider import (
    QubexProvider,
    build_device_topology,
    build_pulse_schedule_timeline_figure,
)

In [2]:
topology = build_device_topology(
    calib_note_path="qubex-config/calibration/calib_note.json",
    params_dir="qubex-config/params",
    name="4Q-DEMO",
    device_id="4Q-DEMO",
)
exp = Experiment(
    system_id="4Q-DEMO-SYS",
    muxes=[0],
    config_dir="qubex-config/config",
    params_dir="qubex-config/params",
    calib_note_path="qubex-config/calibration/calib_note.json",
    calibration_valid_days=10000,
    mock_mode=True,
)

backend_new = QubexProvider.from_experiment(
    exp, device_topology=topology, timing_policy="qiskit"
).get_backend()
backend_legacy = QubexProvider.from_experiment(
    exp, device_topology=topology, timing_policy="legacy_device_gateway"
).get_backend()

# Synthetic provider = local statevector simulator, for the ideal curve.
backend_ideal = QubexProvider(
    num_qubits=4, coupling_map=[(0, 1), (0, 2), (3, 2)]
).get_backend()

RUN_ON_HARDWARE = False
SHOTS = 4096

[qubex.experiment.experiment_context] WARNING: Skew file not found: /Users/orangekame3/src/github.com/orangekame3/qiskit-qubex-provider/examples/qubex-config/config/skew.yaml


date: 2026-06-12 22:52:57


python: 3.13.11


qubex: 1.5.0b4


env: /Users/orangekame3/.cache/uv/builds-v0/.tmpWAqmKh


...

params: /Users/orangekame3/src/github.com/orangekame3/qiskit-qubex-provider/examples/qubex-config/params


chip: 16Q-DEMO (16-qubit demo chip)


qubits: ['Q00', 'Q01', 'Q02', 'Q03']


muxes: ['MUX0']


boxes: ['DEMO2', 'DEMO1']


## The model

A 1D Heisenberg chain under first-order Trotter evolution: start from the
Neel state (|0101⟩), then per layer apply nearest-neighbor
exp(−i θ/2 XX), exp(−i θ/2 YY), exp(−i Δθ/2 ZZ) in an even/odd brickwork
pattern. The chain maps onto the device topology as the physical path
`Q01–Q00–Q02–Q03`. The observable is the central even-site ⟨Z₂⟩, which
oscillates with the total evolution angle θ.

In [3]:
NUM_QUBITS = 4
LAYERS = 6  # matches qiskit-qubex-bench (HEISENBERG_TROTTER_LAYERS)
ANISOTROPY = 1.0
OBS_QUBIT = 2  # central even site


def heisenberg_circuit(total_theta: float) -> QuantumCircuit:
    qc = QuantumCircuit(NUM_QUBITS, NUM_QUBITS)
    for q in range(1, NUM_QUBITS, 2):
        qc.x(q)  # Neel state
    step = total_theta / LAYERS
    for layer in range(LAYERS):
        for q in range(layer % 2, NUM_QUBITS - 1, 2):
            qc.rxx(step, q, q + 1)
            qc.ryy(step, q, q + 1)
            qc.rzz(ANISOTROPY * step, q, q + 1)
    qc.barrier()
    qc.measure(range(NUM_QUBITS), range(NUM_QUBITS))
    return qc


heisenberg_circuit(np.pi / 2).draw(fold=120)

┌────────────┐┌────────────┐                                                  ┌────────────┐┌────────────┐»
q_0: ─────┤0           ├┤0           ├─■────────────────────────────────────────────────┤0           ├┤0           ├»
     ┌───┐│  Rxx(π/12) ││  Ryy(π/12) │ │ZZ(π/12) ┌────────────┐┌────────────┐           │  Rxx(π/12) ││  Ryy(π/12) │»
q_1: ┤ X ├┤1           ├┤1           ├─■─────────┤0           ├┤0           ├─■─────────┤1           ├┤1           ├»
     └───┘├────────────┤├────────────┤           │  Rxx(π/12) ││  Ryy(π/12) │ │ZZ(π/12) ├────────────┤├────────────┤»
q_2: ─────┤0           ├┤0           ├─■─────────┤1           ├┤1           ├─■─────────┤0           ├┤0           ├»
     ┌───┐│  Rxx(π/12) ││  Ryy(π/12) │ │ZZ(π/12) └────────────┘└────────────┘           │  Rxx(π/12) ││  Ryy(π/12) │»
q_3: ┤ X ├┤1           ├┤1           ├─■────────────────────────────────────────────────┤1           ├┤1           ├»
     └───┘└────────────┘└────────────┘                                                  └────────────┘└────────────┘»
c: 4/═══════════════════════════════════════════════════════════════════════════════════════════════════════════════»
                                                                                                                    »
«                                                       ┌────────────┐┌────────────┐                         »
«q_0: ─■────────────────────────────────────────────────┤0           ├┤0           ├─■───────────────────────»
«      │ZZ(π/12) ┌────────────┐┌────────────┐           │  Rxx(π/12) ││  Ryy(π/12) │ │ZZ(π/12) ┌────────────┐»
«q_1: ─■─────────┤0           ├┤0           ├─■─────────┤1           ├┤1           ├─■─────────┤0           ├»
«                │  Rxx(π/12) ││  Ryy(π/12) │ │ZZ(π/12) ├────────────┤├────────────┤           │  Rxx(π/12) │»
«q_2: ─■─────────┤1           ├┤1           ├─■─────────┤0           ├┤0           ├─■─────────┤1           ├»
«      │ZZ(π/12) └────────────┘└────────────┘           │  Rxx(π/12) ││  Ryy(π/12) │ │ZZ(π/12) └────────────┘»
«q_3: ─■────────────────────────────────────────────────┤1           ├┤1           ├─■───────────────────────»
«                                                       └────────────┘└────────────┘                         »
«c: 4/═══════════════════════════════════════════════════════════════════════════════════════════════════════»
«                                                                                                            »
«                               ░ ┌─┐         
«q_0: ──────────────────────────░─┤M├─────────
«     ┌────────────┐            ░ └╥┘┌─┐      
«q_1: ┤0           ├─■──────────░──╫─┤M├──────
«     │  Ryy(π/12) │ │ZZ(π/12)  ░  ║ └╥┘┌─┐   
«q_2: ┤1           ├─■──────────░──╫──╫─┤M├───
«     └────────────┘            ░  ║  ║ └╥┘┌─┐
«q_3: ──────────────────────────░──╫──╫──╫─┤M├
«                               ░  ║  ║  ║ └╥┘
«c: 4/═════════════════════════════╩══╩══╩══╩═
«                                  0  1  2  3

## Compile the same gates under both timing policies

Both backends share the experiment and calibration — the **only**
difference is how the circuit becomes a pulse schedule. To keep the
comparison fair, each policy gets its natural input built from the
**same physical gate sequence**:

- the new policy consumes the ALAP-**scheduled** circuit (it follows the
  transpiler's start times)
- the legacy policy consumes the **unscheduled** physical circuit, as
  the device-gateway flow always did (it ignores start times and emits
  operations sequentially, with barriers around every virtual-Z)

In [4]:
physical = transpile(
    heisenberg_circuit(np.pi / 2),
    backend_new,
    optimization_level=2,
    seed_transpiler=7,
)
# Same gates, plus the schedule (start times and explicit delays).
scheduled = transpile(physical, backend_new, scheduling_method="alap", optimization_level=0)

schedule_new = backend_new.validate(scheduled)[0]
schedule_legacy = backend_legacy.validate(physical)[0]
print(f"qiskit policy : {schedule_new.duration:7.0f} ns")
print(f"legacy policy : {schedule_legacy.duration:7.0f} ns")
print(f"ratio         : {schedule_legacy.duration / schedule_new.duration:.2f}x")

qiskit policy :   23632 ns
legacy policy :   29256 ns
ratio         : 1.24x


In [5]:
build_pulse_schedule_timeline_figure(
    schedule_new, title="timing_policy='qiskit' (parallel brickwork)"
).show()

In [6]:
build_pulse_schedule_timeline_figure(
    schedule_legacy, title="timing_policy='legacy_device_gateway' (serialized)"
).show()

Look at the cross-resonance lanes: under the new policy the even-layer
interactions on `Q00-Q01` and `Q03-Q02` run **simultaneously** in the
first ≈2 µs; under the legacy policy `Q03-Q02` does not start until
`Q00-Q01` (and everything else before it) has finished — the whole
schedule is one long serial chain. On this small star-shaped topology
only one pair of bricks can overlap, which is worth ≈20%; on wider
chains every extra parallel brick widens the gap.

### Pitfall: feeding scheduled circuits to the legacy policy

The modern toolchain wants scheduled circuits (DD insertion and timeline
analysis require them). But a scheduled circuit encodes its idle time as
explicit `delay` instructions computed **assuming parallel execution** —
and the legacy path replays those delays as blanks on top of its serial
schedule, double-counting the idle time:

In [7]:
schedule_legacy_inflated = backend_legacy.validate(scheduled)[0]
print(f"legacy from unscheduled circuit : {schedule_legacy.duration:7.0f} ns")
print(f"legacy from scheduled circuit   : {schedule_legacy_inflated.duration:7.0f} ns  "
      f"({schedule_legacy_inflated.duration / schedule_legacy.duration:.2f}x inflated)")

legacy from unscheduled circuit :   29256 ns
legacy from scheduled circuit   :   62640 ns  (2.14x inflated)


The inflated schedule shows up as mysterious blank stretches between
the CR blocks — idle time that exists only because the delays were
computed for a parallel schedule the legacy path cannot produce. In
other words, the deprecated policy is incompatible with the scheduled
circuits the rest of the ecosystem produces, which is its own reason to
migrate.

## Sweep the evolution angle

The advantage is not a one-off: build the schedules across a θ sweep and
compare total durations. Shorter schedules spend less time exposed to
decoherence — at 6 Trotter layers the legacy schedules reach ≈29 µs,
already comparable to this demo chip's configured T₂ᵉᶜʰᵒ of ≈36 µs.

In [8]:
THETAS = np.linspace(0, 4 * np.pi, 41)  # hardware sweep grid (bench: 0 to 4pi, step pi/10)

physical_circuits = [
    transpile(
        heisenberg_circuit(float(theta)),
        backend_new,
        optimization_level=2,
        seed_transpiler=7,
    )
    for theta in THETAS
]
scheduled_circuits = [
    transpile(c, backend_new, scheduling_method="alap", optimization_level=0)
    for c in physical_circuits
]
durations_new = [backend_new.validate(c)[0].duration for c in scheduled_circuits]
durations_legacy = [backend_legacy.validate(c)[0].duration for c in physical_circuits]

fig = go.Figure()
fig.add_trace(go.Scatter(x=THETAS, y=np.array(durations_legacy) / 1000,
                         name="legacy_device_gateway", line=dict(color="#dc2626")))
fig.add_trace(go.Scatter(x=THETAS, y=np.array(durations_new) / 1000,
                         name="qiskit", line=dict(color="#2563eb")))
fig.update_layout(
    title=("Schedule duration vs evolution angle "
           f"(mean ratio {np.mean(np.array(durations_legacy) / np.array(durations_new)):.2f}x)"),
    xaxis=dict(title="Total evolution angle θ (rad)"),
    yaxis=dict(title="Schedule duration (µs)", rangemode="tozero"),
    width=700, height=420,
)
fig.show()

## Ideal dynamics (offline reference)

The synthetic provider's local simulator gives the noiseless reference
curve for ⟨Z₂⟩(θ) — this is what both hardware variants are trying to
reproduce:

In [9]:
def z_expectation(counts: dict, bit_index: int, shots: int) -> float:
    value = 0
    for key, n in counts.items():
        bit = key.replace(" ", "")[::-1][bit_index]  # clbit 0 = LSB
        value += n if bit == "0" else -n
    return value / shots


# Denser grid than the hardware sweep, so the reference curve is smooth.
THETAS_IDEAL = np.linspace(0, 4 * np.pi, 81)
ideal = []
for theta in THETAS_IDEAL:
    tqc = transpile(heisenberg_circuit(float(theta)), backend_ideal, seed_transpiler=7)
    counts = backend_ideal.run(tqc, shots=SHOTS, seed_simulator=11).result().get_counts()
    ideal.append(z_expectation(counts, OBS_QUBIT, SHOTS))
print("ideal <Z2> range:", round(min(ideal), 2), "to", round(max(ideal), 2),
      f"over {len(THETAS_IDEAL)} points")

ideal <Z2> range: -1.0 to 1.0 over 81 points


## Run on hardware

On a connected setup, run the sweep through both backends — each with
its natural input (scheduled for the new policy, unscheduled for the
legacy one). The expectation: the qiskit-policy points track the ideal
curve better, since every data point spends less time decohering.

In [10]:
measured = None
if RUN_ON_HARDWARE:
    def sweep_z(backend, circuits):
        result = backend.run(circuits, shots=SHOTS).result()
        return [
            z_expectation(result.get_counts(i), OBS_QUBIT, SHOTS)
            for i in range(len(circuits))
        ]

    measured = {
        "qiskit": sweep_z(backend_new, scheduled_circuits),
        "legacy_device_gateway": sweep_z(backend_legacy, physical_circuits),
    }
    print(measured)

In [11]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=THETAS_IDEAL, y=ideal, name="ideal (statevector)",
                         line=dict(color="#475569", dash="dot")))
if measured is not None:
    fig.add_trace(go.Scatter(x=THETAS, y=measured["qiskit"], mode="lines+markers",
                             name="qiskit policy (measured)",
                             marker=dict(color="#2563eb", size=9)))
    fig.add_trace(go.Scatter(x=THETAS, y=measured["legacy_device_gateway"],
                             mode="lines+markers",
                             name="legacy policy (measured)",
                             marker=dict(color="#dc2626", size=9)))
fig.update_layout(
    title="Heisenberg dynamics: central even-site ⟨Z₂⟩ vs θ",
    xaxis=dict(title="Total evolution angle θ (rad)"),
    yaxis=dict(title="⟨Z₂⟩", range=[-1.05, 1.05]),
    width=700, height=420,
)
fig.show()

## Wrap-up

Two concrete reasons to move off the deprecated device-gateway
scheduling, both visible at the pulse level:

- **Parallelism**: the legacy path serializes the brickwork — ≈1.2× more
  wall-clock per shot on this 4-qubit chain, and the gap grows with
  every brick that could have run in parallel on a wider device.
- **Ecosystem compatibility**: scheduled circuits (required for DD and
  timeline tooling) silently inflate legacy schedules by >2×, because
  their delays assume the parallelism the legacy path cannot express.

Extensions, following
[qiskit-qubex-bench](https://github.com/orangekame3/qiskit-qubex-bench):
more layers and chain lengths, DD on the idle windows
([dd-demonstration.ipynb](dd-demonstration.ipynb)), and Aer noise-model
baselines. See [tutorial.ipynb](tutorial.ipynb) for the pipeline
walkthrough.